# 09 — Mistral Small (LLM locale via ollama)

Terzo LLM locale del benchmark (vedi il notebook [07_qwen.ipynb](07_qwen.ipynb) per
l'impostazione generale). Lo slug del metodo resta `mistral`, ma **attenzione all'identità del
modello**, diversa tra le due corse:

- la **corsa originale di Federica** (LM Studio) invocava
  **`mistralai/mistral-7b-instruct-v0.3`** — nonostante il notebook di origine si chiamasse
  *mistral-small3-7B*;
- la **corsa committata** (questo notebook via ollama, 2026-07-16) usa il tag **`mistral-small`**,
  cioè **Mistral Small (~24B parametri, quantizzazione Q4_K_M)** — un modello più recente e
  ~3× più grande.

I punteggi committati di `mistral` **non sono quindi confrontabili con la corsa originale né,
per classe dimensionale, con qwen (7B) e gemma (E4B)**: nel leggere il confronto del notebook
[05_confronto.ipynb](05_confronto.ipynb) va tenuto conto che questo metodo parte da un modello
molto più capiente degli altri due LLM.

Altra precisazione sulla corsa originale: il prompt "di sistema" era stato inviato come
messaggio `user` (Mistral 7B v0.3 non ha un vero ruolo system); qui usiamo il ruolo `system`
standard, che il template chat di ollama gestisce da sé.

## Provenienza dei risultati committati e ripresa

I file committati (`results/summaries/mistral_sample.tsv` e le metriche) provengono dalla
**corsa di questo notebook via ollama** (100/100 esempi, 2026-07-16), che ha rigenerato da
zero i riassunti sostituendo quelli importati dalla corsa originale di Federica via LM Studio
(Mac M4, 2026-07-16; i CSV originali restano archiviati in `notebooks/llm/`). Rispetto alla
corsa originale ci sono quattro differenze deliberate:

- **il modello è diverso**: Mistral Small (~24B, tag ollama `mistral-small`) invece di
  `mistralai/mistral-7b-instruct-v0.3` (vedi sopra);
- il documento passa da `su.prepara_documento` (separatore `` ||||| `` → newline), mentre la
  corsa originale inviava il testo grezzo con il separatore incluso;
- niente `extra_body={"enable_thinking": False}` (parametro specifico di LM Studio; ollama
  lo ignorerebbe);
- il prompt di sistema usa il ruolo `system` (nell'originale era un messaggio `user`).

⚠️ **Attenzione alla ripresa**: il ciclo condiviso salta i `row_id` già presenti nel TSV, quindi
rieseguire la generazione su un TSV esistente **aggiunge solo le righe mancanti** — utile per
completare una corsa ollama interrotta, ma da evitare su un file prodotto da un backend, un
modello o una configurazione diversi (es. un re-import della corsa LM Studio), perché
mescolerebbe le due corse nello stesso file. In quel caso **eliminare prima**
`results/summaries/mistral_sample.tsv` e rigenerare tutto (rieseguendo poi la valutazione).

In [1]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer
try:
    import openai  # noqa: F401
except ImportError:
    %pip install openai

In [ ]:
# --- Configurazione ---------------------------------------------------------
import summ_utils as su

METODO     = 'mistral'
SCOPE      = 'sample'    # questo notebook gira solo sul campione
N_SAMPLES  = 100         # deve combaciare con il campione creato dal notebook 00
SEED       = 42
LIMIT      = None        # es. 3 per uno smoke test rapido; None = tutti
MODELLO    = 'mistral-small'  # Mistral Small ~24B — NON il 7B v0.3 della corsa originale (vedi sopra)
OLLAMA_URL = 'http://localhost:11434/v1'   # endpoint OpenAI-compatibile di ollama

MAX_TOKENS  = 300         # come la corsa LM Studio originale
TEMPERATURE = 0.3
PROMPT_SYSTEM = ('You are a helpful assistant that summarizes news articles '
                 'from different sources concisely.')
PROMPT_USER   = (''
                 'Summarize the following document into a comprehensive summary: {documento}')
ETICHETTA   = 'Mistral '
NOTE_CONFIG = ('prompt zero-shot in inglese; modello Mistral Small ~24B (tag ollama '
               'mistral-small), diverso dal mistral-7b-instruct-v0.3 della corsa '
               'originale LM Studio (il cui notebook si chiamava "mistral-small3-7B")')

BASE = su.trova_base_dir()
P    = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH    = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'

print(f'Modello : {MODELLO} via {OLLAMA_URL}')
print(f'Output  : {OUT_PATH}')

## Generazione dei riassunti

Il modello gira **fuori dal notebook**, nel server locale di ollama: qui usiamo solo il client
`openai` puntato all'endpoint OpenAI-compatibile (`http://localhost:11434/v1`). Prerequisiti:

```bash
ollama pull mistral-small   # Mistral Small ~24B — NON il 7B della corsa originale (vedi sopra)
ollama serve                # se il servizio non e' gia' attivo
```

Il ciclo e la scrittura incrementale (con ripresa) sono quelli condivisi di `summ_utils`; una
risposta vuota del modello solleva un'eccezione, così la riga viene registrata come errore e
**non** scritta nel TSV (ritentabile alla corsa successiva).

In [3]:
from openai import OpenAI

client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')  # la chiave e' ignorata da ollama

def genera(documento):
    risposta = client.chat.completions.create(
        model=MODELLO,
        messages=[{'role': 'system', 'content': PROMPT_SYSTEM},
                  {'role': 'user', 'content': PROMPT_USER.format(documento=documento)}],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE)
    scelta = risposta.choices[0]
    contenuto = scelta.message.content
    if not contenuto or not contenuto.strip():
        # solleva -> il ciclo condiviso registra l'errore e NON scrive la riga
        raise RuntimeError(f'risposta vuota (finish_reason={scelta.finish_reason})')
    return contenuto.strip()

esempi    = su.carica_campione(SAMPLE_PATH)
scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(esempi, scrittore, genera, limit=LIMIT,
                                etichetta=ETICHETTA)
scrittore.chiudi()

Mistral [1] media 73.9 s/esempio (saltati 0 gia' fatti)
Mistral [2] media 65.2 s/esempio (saltati 0 gia' fatti)
Mistral [3] media 62.2 s/esempio (saltati 0 gia' fatti)
Mistral [10] media 57.6 s/esempio (saltati 0 gia' fatti)
Mistral [20] media 58.7 s/esempio (saltati 0 gia' fatti)
Mistral [30] media 56.6 s/esempio (saltati 0 gia' fatti)
Mistral [40] media 53.8 s/esempio (saltati 0 gia' fatti)
Mistral [50] media 54.3 s/esempio (saltati 0 gia' fatti)
Mistral [60] media 53.3 s/esempio (saltati 0 gia' fatti)
Mistral [70] media 52.7 s/esempio (saltati 0 gia' fatti)
Mistral [80] media 52.7 s/esempio (saltati 0 gia' fatti)
Mistral [90] media 52.4 s/esempio (saltati 0 gia' fatti)
Mistral [100] media 52.3 s/esempio (saltati 0 gia' fatti)
Mistral Completato: 100 nuovi, 0 saltati, 0 errori, 5226 s totali


## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti. Metriche ROUGE-1/2/L
(F1, precisione, recall), BLEU e METEOR con normalizzazione identica per tutti i metodi del
benchmark. Output: `results/metrics/mistral_sample_per_example.csv` e `…_aggregate.json`.

⚠️ La valutazione misura il TSV salvato **qualunque ne sia l'origine**, mentre il JSON aggregato
riporta la configurazione dichiarata qui sotto: oggi le due cose coincidono (il TSV committato
è la corsa ollama di questo notebook), ma se il TSV venisse re-importato da un'altra corsa
(es. LM Studio via `scripts/import_llm_results.py`, che scrive la propria provenienza nel JSON),
rieseguire questa cella sovrascriverebbe quella provenienza. Rieseguirla solo su un TSV
generato con la configurazione qui sopra.

In [4]:
import json

riassunti   = su.carica_riassunti(OUT_PATH)
riferimenti = su.carica_campione(SAMPLE_PATH)
config = {'modello': MODELLO,
          'backend': 'ollama (endpoint OpenAI-compatibile)',
          'max_tokens': MAX_TOKENS, 'temperature': TEMPERATURE,
          'prompt_system': PROMPT_SYSTEM, 'prompt_user': PROMPT_USER,
          'note': NOTE_CONFIG}

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))
print('\nMedie per split:')
for split, valori in aggregato['per_split'].items():
    print(f"  {split:5s} (n={valori['n_esempi']}): ROUGE-1 F1 = {valori['rouge1_f1']:.3f}")

c:\Users\antonio.girasella\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Metriche per-esempio : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\mistral_sample_per_example.csv (100 righe)
Metriche aggregate   : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\mistral_sample_aggregate.json
{
  "rouge1_f1": 0.34430871260396523,
  "rouge1_p": 0.3693922371150044,
  "rouge1_r": 0.33427749898578474,
  "rouge2_f1": 0.10215909880851372,
  "rouge2_p": 0.11231166326936189,
  "rouge2_r": 0.0986245171097246,
  "rougeL_f1": 0.18053141189026886,
  "rougeL_p": 0.20032120910329967,
  "rougeL_r": 0.17248376737265297,
  "bleu": 0.06858631496401871,
  "meteor": 0.36982371931939084,
  "parole_generate": 193.35
}

Medie per split:
  test  (n=8): ROUGE-1 F1 = 0.346
  train (n=81): ROUGE-1 F1 = 0.341
  val   (n=11): ROUGE-1 F1 = 0.368


## Ispezione qualitativa

In [5]:
su.mostra_esempi(su.carica_campione(SAMPLE_PATH), riassunti, quanti=2)

--- row_id=425 (split=train) ---
GENERATO   : A Republican state Senate candidate in Connecticut, Ed Charamut, sent out a campaign mailer depicting his Democratic opponent, Matthew Lesser (who is Jewish), holding wads of cash with an altered face that critics described as anti-Semitic. The mailer featured a tagline suggesting Lesser would take everything voters had worked for.

The mailer drew widespread condemnation, especially in the context of recent anti-Semitic attacks, including the Pittsburgh synagogue massacre. Critics pointed out t
RIFERIMENTO: Just days after the slaying of 11 Jewish congregants at a Pittsburgh synagogue, a GOP candidate for a state Senate seat in Connecticut is accused of sending a mailer using an "age-old anti-Semitic trope." The ad sent out by Ed Charamut includes what the Washington Post calls a "money-grubbing" picture (here) of smiling opponent Matt Lesser, clutching $100 bills with a "crazed look in his eyes." Lesser says the original image of him was 